**Author:** Salvador Navas  
**Model:** SFINCS (Super-Fast INundation of CoastS) — Deltares  
**Sources:** [SFINCS docs](https://sfincs.readthedocs.io) · [hydromt_sfincs](https://deltares.github.io/hydromt_sfincs) · [Deltares GitHub](https://github.com/Deltares/SFINCS)

### SFINCS — Automated flood modelling with `pyhydra` + `hydromt_sfincs`

SFINCS is an open-source reduced-complexity 2D flood model (overland + coastal).  
`hydromt_sfincs` is the Deltares Python interface for building, running and post-processing SFINCS models.

**Install:**
```bash
pip install hydromt_sfincs xarray rasterio
# SFINCS binary (Linux): conda install -c conda-forge sfincs
```

**Key role in HYDRA** — hybrid downscaling requires running **many hydraulic simulations**
(one per climate-change scenario × return period). The automation here (`setup_sfincs_model`,
`run_sfincs`) is designed to be called inside a loop over the ensemble.

Workflow:
1. Build a fluvial flood-plain domain with a synthetic DEM
2. Inspect model structure with hydromt_sfincs
3. Define the T100 design hydrograph as discharge BC
4. Write and run SFINCS (binary via `pyhydra.modeling.hydraulic.sfincs`)
5. Post-process: flood depth, extent, volume
6. Illustrative climate-change ensemble - discharge scaling with documented source


In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import shutil
import numpy as np
import pandas as pd
import xarray as xr
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from pathlib import Path
from datetime import datetime, timedelta

from pyhydra.modeling.hydraulic.sfincs import setup_sfincs_model, run_sfincs


---
## 1. Build a synthetic fluvial floodplain domain

We create a 6 km × 4 km domain entirely in Python — no download required.
The DEM has a river channel (5 m deep) running through the centre with
gently sloping floodplains on both sides.  
This synthetic domain is used to demonstrate the full automation workflow.


In [2]:
from hydromt_sfincs import SfincsModel

# ── Model paths ───────────────────────────────────────────────────────────────
# /workspace/data is read-only in Azure/Jupyter; model runs are written per session.
RUNTIME_ROOT = Path(os.environ.get('HYDRA_RUNTIME_DIR', Path.cwd() / '.hydra_runtime'))
SFINCS_DIR  = RUNTIME_ROOT / 'sfincs'
BASE_DIR    = str(SFINCS_DIR / 'base_model')
# SFINCS binary: install via  conda install -c conda-forge sfincs
# or build from source: github.com/Deltares/SFINCS
SFINCS_EXE  = shutil.which('sfincs') or '/workspace/tools/sfincs/sfincs'

# ── Grid parameters ───────────────────────────────────────────────────────────
DX, DY   = 50.0, 50.0       # cell size (m)
NMAX     = 80               # rows
MMAX     = 120              # cols
EPSG     = 32630            # UTM zone 30N (generic)
X0, Y0   = 0.0, 0.0        # origin

SFINCS_DIR.mkdir(parents=True, exist_ok=True)

# ── Build synthetic DEM ───────────────────────────────────────────────────────
rows = np.arange(NMAX)
cols = np.arange(MMAX)
y    = Y0 + rows * DY + DY / 2
x    = X0 + cols * DX + DX / 2
xx, yy = np.meshgrid(x, y)

# HEC-RAS-compatible geometry (same parameters as HEC_RAS notebook)
cy = Y0 + NMAX * DY / 2
ch_half_m = 50.0   # channel half-width (m) → 100 m total, 2 cells at DY=50 m
ch_mask   = np.abs(yy - cy) < ch_half_m
# Bank at 0.5 m; valley slopes 0.002 m/m from bank edge (same as HEC-RAS)
fp_elev   = 0.5 + 0.002 * np.maximum(0.0, np.abs(yy - cy) - ch_half_m)
# Channel bottom at −2.5 m → full-bank capacity ≈ 242 m³/s; T100 (600 m³/s) overflows ~358 m³/s
zb        = np.where(ch_mask, -2.5, fp_elev)
# Longitudinal slope: 0.2 m/km upstream→downstream
zb += (X0 + MMAX * DX - xx) * 2e-4

dem = xr.DataArray(
    zb,
    dims=['y', 'x'],
    coords={'y': y, 'x': x},
    name='zb',
    attrs={'long_name': 'bed elevation', 'units': 'm'},
)

# ── Build SFINCS model ────────────────────────────────────────────────────────
if Path(BASE_DIR, 'sfincs.inp').exists():
    shutil.rmtree(BASE_DIR)

sf = SfincsModel(root=BASE_DIR, mode='w')
sf.setup_grid(
    x0=X0, y0=Y0, dx=DX, dy=DY,
    nmax=NMAX, mmax=MMAX,
    rotation=0, epsg=EPSG,
)

# Write DEM into model grid (interpolate DataArray onto SFINCS grid)
dem_file = str(SFINCS_DIR / 'synthetic_dem.tif')
dem.rio.set_spatial_dims('x', 'y').rio.write_crs(f'EPSG:{EPSG}').rio.to_raster(dem_file)
sf.setup_dep([{'elevtn': dem_file, 'zmin': -10}])
sf.setup_mask_active(zmin=-5.0)

# Simulation window: 2024-01-01  00:00 → 2024-01-03  00:00 (48 h)
TREF   = '20240101 000000'
TSTART = '20240101 000000'
TSTOP  = '20240103 000000'
sf.set_config('tref',   TREF)
sf.set_config('tstart', TSTART)
sf.set_config('tstop',  TSTOP)
sf.set_config('dtmax',  30)    # max time step (s)

sf.write()
print('✓ Base SFINCS model written →', BASE_DIR)
print(f'  Grid : {MMAX} × {NMAX} cells  ({DX} m resolution)')
print(f'  Size : {MMAX*DX/1000:.1f} km × {NMAX*DY/1000:.1f} km')
print(f'  CRS  : EPSG:{EPSG}')


✓ Base SFINCS model written → /workspace/sessions/.hydra_runtime/sfincs/base_model
  Grid : 120 × 80 cells  (50.0 m resolution)
  Size : 6.0 km × 4.0 km
  CRS  : EPSG:32630


SHA256 hash of downloaded file: 32de5b95c171628547f303d7f65d53cbb1b9da9af4834717c8efff93fe55aad4
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.
Untarring contents of '/root/.hydromt_data/artifact_data/v0.0.9/data.tar.gz' to '/root/.hydromt_data/artifact_data/v0.0.9/data.tar'


---
## 2. Inspect model structure with hydromt_sfincs


In [3]:
# Re-open in read mode to inspect
sf_r = SfincsModel(root=BASE_DIR, mode='r')
sf_r.read()

print('Grid dims :', sf_r.grid.dims)
print('Grid vars :', list(sf_r.grid.data_vars))
print('Config:')
for k in ['tref', 'tstart', 'tstop', 'dx', 'dy', 'epsg', 'dtmax']:
    print(f'  {k:10s} = {sf_r.config.get(k, "n/a")}')

zb_var = 'dep' if 'dep' in sf_r.grid else 'zb'
zb_da  = sf_r.grid[zb_var]
zb_v   = zb_da.values        # (NMAX, MMAX)
x_c    = zb_da.coords['x'].values
y_c    = zb_da.coords['y'].values

# Detrend: remove longitudinal slope to show cross-section structure clearly
long_slope  = 2e-4
zb_detrend  = zb_v - (X0 + MMAX * DX - x_c[np.newaxis, :]) * long_slope

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: detrended 2D bed elevation map
x_edges = np.concatenate([x_c - DX/2, [x_c[-1] + DX/2]])
y_edges = np.concatenate([y_c - DY/2, [y_c[-1] + DY/2]])
im = axes[0].pcolormesh(x_edges, y_edges, zb_detrend,
                        cmap='RdYlGn', shading='flat', vmin=-3, vmax=4)
plt.colorbar(im, ax=axes[0], label='Bed elevation — detrended (m)')
axes[0].set_title('Bed elevation (detrended)\nChannel: 100 m wide · bank 0.5 m · bottom −2.5 m · slope 0.002 m/m')
axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')
axes[0].set_aspect('equal')

# Panel 2: cross-section through channel centre (midpoint column)
mid_col = MMAX // 2
zb_xs   = zb_detrend[:, mid_col]
cy_m    = Y0 + NMAX * DY / 2
axes[1].fill_between(y_c, zb_xs, -4.0, alpha=0.35, color='saddlebrown', label='DEM profile')
axes[1].plot(y_c, zb_xs, 'k-', lw=1.5)
axes[1].axhline(0.5,  ls='--', color='steelblue', lw=1.5, label='Bank elevation (0.5 m)')
axes[1].axhline(-2.5, ls=':',  color='peru',      lw=1.2, label='Channel bottom (−2.5 m)')
axes[1].axvline(cy_m, ls=':',  color='gray',      lw=1,   label=f'Channel centre ({cy_m:.0f} m)')
axes[1].set_ylim(-3.5, 5)
axes[1].set(xlabel='y (m)', ylabel='Elevation (m)',
            title='Cross-section at channel centre\n'
                  'Capacity (Manning) ≈ 242 m³/s  |  Q_T100 = 600 m³/s → 358 m³/s overbank')
axes[1].legend(fontsize=8); axes[1].grid(True, alpha=0.3)
plt.tight_layout()


Grid dims : FrozenMappingWarningOnValuesAccess({'y': 80, 'x': 120})
Grid vars : ['msk', 'dep']
Config:
  tref       = 2024-01-01 00:00:00
  tstart     = 2024-01-01 00:00:00
  tstop      = 2024-01-03 00:00:00
  dx         = 50.0
  dy         = 50.0
  epsg       = 32630
  dtmax      = 30


---
## 3. T100 design flood hydrograph

The upstream boundary condition is a triangular flood wave derived from
a 100-year return-period frequency analysis (placeholder values here).  
For the climate-change ensemble, this hydrograph is scaled by per-scenario
delta factors from the hydrological model (HMS or SWAT+).


In [4]:
Q_PEAK   = 600.0                # m³/s  (T100 reference peak flow)
DT_MIN   = 5                    # time step (min)
N_STEPS  = int(48 * 60 / DT_MIN)
T_PEAK_H = 12.0                 # time to peak (h)

t_h    = np.linspace(0, 48, N_STEPS)
q_vals = Q_PEAK * (t_h / T_PEAK_H) * np.exp(1.0 - t_h / T_PEAK_H)
q_vals = np.maximum(q_vals, 0.0)

TSTART_DT = datetime(2024, 1, 1)
t_idx     = pd.date_range(TSTART_DT, periods=N_STEPS, freq=f'{DT_MIN}min')
Q_ref     = pd.Series(q_vals, index=t_idx, name='Q_m3s')

fig, ax = plt.subplots(figsize=(11, 3))
Q_ref.plot(ax=ax, color='steelblue', lw=2)
ax.axhline(Q_PEAK, color='red', ls='--', alpha=0.6, label=f'Q_peak = {Q_PEAK} m³/s')
ax.fill_between(Q_ref.index, Q_ref, alpha=0.15, color='steelblue')
ax.set(ylabel='Q (m³/s)', xlabel='', title='T100 reference discharge hydrograph')
ax.legend()
plt.tight_layout()

print(f'Duration   : 48 h  ({N_STEPS} steps × {DT_MIN} min)')
print(f'Peak flow  : {Q_PEAK:.0f} m³/s  at t = {T_PEAK_H:.0f} h')
print(f'Volume     : {Q_ref.sum() * DT_MIN * 60 / 1e6:.2f} Mm³')


Duration   : 48 h  (576 steps × 5 min)
Peak flow  : 600 m³/s  at t = 12 h
Volume     : 63.91 Mm³


---
## 4. Configure model: discharge + output + Manning roughness

`pyhydra.modeling.hydraulic.sfincs.setup_sfincs_model` configures a complete
SFINCS run directory from a basin polygon, DEM and discharge time series.  
Here we use the synthetic domain built above. For real catchments, pass
the actual basin GeoDataFrame and DEM GeoTIFF.


In [5]:
from shapely.geometry import box as shapely_box
from pyhydra.modeling.hydraulic.sfincs import write_manning_wl_boundary

RUN_DIR = str(SFINCS_DIR / 'run_T100')

# Domain polygon (full grid extent)
domain_box = shapely_box(X0, Y0, X0 + MMAX * DX, Y0 + NMAX * DY)
basin_gdf  = gpd.GeoDataFrame(geometry=[domain_box], crs=f'EPSG:{EPSG}')

# Upstream discharge point — left edge, centreline
src_x = X0 + DX / 2
src_y = Y0 + NMAX * DY / 2
src_gdf = gpd.GeoDataFrame(
    {'Id': [0]},
    geometry=gpd.points_from_xy([src_x], [src_y]),
    crs=f'EPSG:{EPSG}',
)
src_gdf.index = [0]

discharge_series = Q_ref.to_frame(name=0)

model = setup_sfincs_model(
    basin_geom       = domain_box,
    dem_path         = dem_file,
    output_dir       = RUN_DIR,
    discharge_series = discharge_series,
    src_points       = src_gdf,
    crs              = EPSG,
    resolution       = DX,
    manning          = 0.035,
    tref             = TREF,
    tstart           = TSTART,
    tstop            = TSTOP,
    dt_max           = 30.0,
    z_min            = -5.0,
)
model.write()

# Downstream WL boundary: Manning rating curve (msk=2 + bnd/bzs files)
# Must be called AFTER model.write() — overwrites the msk=3 default
write_manning_wl_boundary(
    model_dir        = RUN_DIR,
    discharge_series = discharge_series,
    ch_w             = 2 * ch_half_m,
    ch_zb            = -2.5,
    mann_n           = 0.035,
    slope            = 2e-4,
    tref             = TREF,
)
print('✓ T100 run directory ready →', RUN_DIR)


  src: 1 points, dis: 576 time steps
✓ SFINCS model set up → /workspace/sessions/.hydra_runtime/sfincs/run_T100
  WL boundary: 80 pts at x=5975 m · WL_max=2.75 m
✓ T100 run directory ready → /workspace/sessions/.hydra_runtime/sfincs/run_T100


---
## 5. SFINCS execution — Docker (from host) and analytical validation

SFINCS runs as a standalone binary compiled for `linux/amd64`.
Inside the Jupyter container (also `linux/amd64`) there is no Docker socket,
so the binary is invoked from the **host machine**:

```bash
# Run T100 simulation — execute from the host terminal:
docker run --rm --platform=linux/amd64 \
    -v <path-to-HYDRA>/data/sfincs/run_T100:/data \
    deltares/sfincs-cpu:latest
```

The downstream boundary is prescribed as a **water-level time series** (Manning's
normal depth for each Q(t) of the hydrograph), written automatically to
`sfincs.bnd` + `sfincs.bzs` during model setup.  This gives a **rating-curve
exit condition** and allows the 2D flood to develop across the floodplain.

Results are stored in `sfincs_map.nc`.  The cell below reads the pre-computed
output and compares it against Manning's analytical steady-state solution.


In [6]:
# ── Manning analytical reference (steady uniform flow) ───────────────────────
from scipy.optimize import brentq
import xarray as xr

CH_HALF  = ch_half_m          # 50 m → 100 m total channel width
CH_W     = 2 * CH_HALF
CH_ZB    = -2.5               # channel bottom at downstream end (m)
BANK_Z   = 0.5                # bank elevation (m)
MANN_N   = 0.035
SLOPE    = 2e-4               # longitudinal slope (m/m)

def channel_Q(yn, W=CH_W, n=MANN_N, S=SLOPE):
    A = W * yn
    R = A / (W + 2 * yn)
    return (A / n) * R ** (2 / 3) * S ** 0.5

def manning_yn(Q, W=CH_W, n=MANN_N, S=SLOPE):
    if Q < 1.0:
        return 0.05
    return brentq(lambda yn: channel_Q(yn, W, n, S) - Q, 0.01, 60)

yn_T100   = manning_yn(Q_PEAK)
WSEL_T100 = CH_ZB + yn_T100
Q_fb      = channel_Q(BANK_Z - CH_ZB)

print('── Manning analytical (T100) ──')
print(f'  Channel  : W={CH_W:.0f} m · zb={CH_ZB} m · bank={BANK_Z} m')
print(f'  Full-bank: {Q_fb:.0f} m³/s  (S={SLOPE:.0e}, n={MANN_N})')
print(f'  T100     : Q={Q_PEAK:.0f} m³/s → yn={yn_T100:.2f} m → WSEL={WSEL_T100:.2f} m')
print()

# ── Load DEM from base model ──────────────────────────────────────────────────
sf_r2  = SfincsModel(root=BASE_DIR, mode='r')
sf_r2.read()
zb_var = 'dep' if 'dep' in sf_r2.grid else 'zb'
zb_v   = sf_r2.grid[zb_var].values          # (NMAX, MMAX)
msk_v  = sf_r2.grid['msk'].values if 'msk' in sf_r2.grid else np.ones_like(zb_v)
x_c    = sf_r2.grid[zb_var].coords['x'].values
y_c    = sf_r2.grid[zb_var].coords['y'].values

# Detrend DEM (remove longitudinal slope) for uniform-flow comparison
zb_dt  = zb_v - (X0 + MMAX * DX - x_c[np.newaxis, :]) * SLOPE

# Manning static inundation (equilibrium / worst-case)
depth_manning = np.maximum(0.0, WSEL_T100 - zb_dt)
depth_manning = np.where(msk_v > 0, depth_manning, 0.0)
area_manning  = int(np.sum(depth_manning > 0.30)) * (DX / 1000) ** 2
print(f'  Manning flood area : {area_manning:.2f} km²  (uniform steady flow)')

# ── SFINCS 2D simulation — Docker results ─────────────────────────────────────
print()
print('── SFINCS 2D (dynamic routing) ──')
print('  HOST command (run before this notebook):')
print(f'    docker run --rm --platform=linux/amd64 \\')
t100_dir = str(SFINCS_DIR / 'run_T100')
print(f'      -v {t100_dir}:/data deltares/sfincs-cpu:latest')
print()

sfincs_nc = SFINCS_DIR / 'run_T100' / 'sfincs_map.nc'
HAS_SFINCS = sfincs_nc.exists()

if HAS_SFINCS:
    ds_sf     = xr.open_dataset(str(sfincs_nc))
    hmax_sf   = ds_sf['hmax'].values            # (timemax, n, m)
    peak_sf   = np.nanmax(hmax_sf, axis=0)      # peak over timemax
    depth_sfincs = np.where(msk_v > 0, np.nan_to_num(peak_sf), 0.0)
    area_sfincs  = int(np.sum(depth_sfincs > 0.30)) * (DX / 1000) ** 2
    maxd_sfincs  = float(np.nanmax(depth_sfincs))

    print(f'  SFINCS flood area : {area_sfincs:.2f} km²  (dynamic 48 h simulation)')
    print(f'  Max flood depth   : {maxd_sfincs:.2f} m')
    print()
    print('  Comparison — Manning (steady) vs. SFINCS (dynamic):')
    print(f'    Manning : {area_manning:.2f} km²  → equilibrium upper bound')
    print(f'    SFINCS  : {area_sfincs:.2f} km²  → dynamic routing (WL boundary = Manning rating curve)')
    print(f'    Ratio   : {area_sfincs/area_manning:.2f}  (dynamic routing < steady: incomplete flood development during 48h)')

    depth_T100 = depth_sfincs          # primary for Cell 6 visualisation
else:
    print('  sfincs_map.nc not found — run SFINCS from the host first.')
    print('  Falling back to Manning static inundation.')
    depth_T100   = depth_manning
    area_sfincs  = None
    depth_sfincs = None


── Manning analytical (T100) ──
  Channel  : W=100 m · zb=-2.5 m · bank=0.5 m
  Full-bank: 243 m³/s  (S=2e-04, n=0.035)
  T100     : Q=600 m³/s → yn=5.25 m → WSEL=2.75 m

  Manning flood area : 12.60 km²  (uniform steady flow)

── SFINCS 2D (dynamic routing) ──
  HOST command (run before this notebook):
    docker run --rm --platform=linux/amd64 \
      -v /workspace/sessions/.hydra_runtime/sfincs/run_T100:/data deltares/sfincs-cpu:latest

  sfincs_map.nc not found — run SFINCS from the host first.
  Falling back to Manning static inundation.


---
## 6. Post-process: flood maps — SFINCS 2D vs. Manning reference

Three-panel map:

| Panel | Content |
|-------|---------|
| 1 — DEM | Detrended bed elevation (longitudinal slope removed) |
| 2 — SFINCS | Peak flood depth from `sfincs_map.nc` when available; otherwise labelled Manning fallback |
| 3 — Manning | Steady-state normal depth (analytical upper bound) |

When SFINCS output is available, the difference from Manning reflects the **transient flood wave**:
Manning assumes the entire channel has reached uniform flow simultaneously,
while SFINCS resolves the actual propagation upstream from the WL boundary.


In [7]:
RES_M = DX

# X/Y grid edges for pcolormesh
x_arr = np.linspace(X0, X0 + MMAX * DX, MMAX + 1)
y_arr = np.linspace(Y0, Y0 + NMAX * DY, NMAX + 1)
x_1d  = np.linspace(X0 + DX/2, X0 + MMAX*DX - DX/2, MMAX)
zb_detrend = zb_v - (X0 + MMAX*DX - x_1d[np.newaxis, :]) * 2e-4

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Panel 1: Detrended DEM ────────────────────────────────────────────────────
im0 = axes[0].pcolormesh(x_arr, y_arr,
    np.ma.array(zb_detrend, mask=msk_v == 0),
    cmap='RdYlGn', vmin=-3, vmax=4, shading='flat')
plt.colorbar(im0, ax=axes[0], label='Bed elevation — detrended (m)')
axes[0].set_title('Bed Elevation — Synthetic DEM\n(100 m channel, bank 0.5 m, slope 0.002 m/m)')
axes[0].set_aspect('equal'); axes[0].set_xlabel('x (m)'); axes[0].set_ylabel('y (m)')

# ── Panel 2: SFINCS peak flood depth ─────────────────────────────────────────
n_sf = int(np.sum(depth_T100 > 0.3))
a_sf = n_sf * (RES_M / 1000) ** 2
axes[1].pcolormesh(x_arr, y_arr, np.ma.array(zb_v, mask=msk_v == 0),
    cmap='Greys_r', shading='flat', alpha=0.35, vmin=-2, vmax=8)
dm_masked = np.ma.array(depth_T100, mask=(depth_T100 <= 0.05) | (msk_v == 0))
im1 = axes[1].pcolormesh(x_arr, y_arr, dm_masked,
    cmap='Blues', vmin=0, vmax=5, shading='flat', alpha=0.9)
plt.colorbar(im1, ax=axes[1], label='Peak flood depth (m)')
label_sf = 'SFINCS dynamic' if HAS_SFINCS else 'Manning fallback'
axes[1].set_title(f'Peak Flood Depth — T100 ({label_sf})\n'
                  f'Flood area = {a_sf:.2f} km²  |  max depth = {depth_T100.max():.2f} m')
axes[1].set_aspect('equal'); axes[1].set_xlabel('x (m)')

# ── Panel 3: Manning steady-state reference ───────────────────────────────────
a_mn = int(np.sum(depth_manning > 0.3)) * (RES_M / 1000) ** 2
axes[2].pcolormesh(x_arr, y_arr, np.ma.array(zb_v, mask=msk_v == 0),
    cmap='Greys_r', shading='flat', alpha=0.35, vmin=-2, vmax=8)
dm2_masked = np.ma.array(depth_manning, mask=(depth_manning <= 0.05) | (msk_v == 0))
im2 = axes[2].pcolormesh(x_arr, y_arr, dm2_masked,
    cmap='YlOrRd', vmin=0, vmax=5, shading='flat', alpha=0.9)
plt.colorbar(im2, ax=axes[2], label='Peak flood depth (m)')
axes[2].set_title(f'Manning Reference (steady uniform flow)\n'
                  f'Flood area = {a_mn:.2f} km²  |  WSEL = {WSEL_T100:.2f} m')
axes[2].set_aspect('equal'); axes[2].set_xlabel('x (m)')

plt.suptitle(f'T100 Flood Inundation — SFINCS vs Manning  (Q={Q_PEAK:.0f} m³/s)',
             fontsize=12, y=1.02)
plt.tight_layout()


---
## 7. Climate-change ensemble — automated multi-simulation loop

This is the **core workflow for hybrid downscaling**: each GCM-RCM scenario
produces a discharge delta factor from the hydrological model (HEC-HMS / SWAT+),
which is applied to the reference hydrograph and fed into SFINCS.

The loop below:
1. Scales the T100 hydrograph by the CC delta factor
2. Calls `setup_sfincs_model` to write a new run directory
3. Calls `run_sfincs` to execute the binary
4. Reads `sfincs_map.nc` and extracts flood area + max depth

In practice, `delta_factors` come from the climate-change runs in the  
HEC_HMS or SWAT notebooks (ratio of future-to-reference peak discharge per scenario).


In [ ]:
# Illustrative CC delta factors. Replace with a documented CSV for real studies.
DELTA_SOURCE = os.getenv('HYDRA_SFINCS_DELTA_CSV') or 'illustrative_hydraulic_sensitivity'
CC_SCENARIOS = {
    'MIROC6_ssp245_NF'        : 1.06,
    'MIROC6_ssp585_NF'        : 1.12,
    'MPI-ESM1-2-LR_ssp245_NF' : 0.97,
    'MPI-ESM1-2-LR_ssp585_NF' : 1.04,
    'CNRM-CM6-1_ssp245_NF'    : 1.08,
    'CNRM-CM6-1_ssp585_MF'    : 1.18,
    'ACCESS-CM2_ssp245_NF'    : 1.03,
    'ACCESS-CM2_ssp585_MF'    : 1.09,
    'MRI-ESM2-0_ssp585_FF'    : 1.22,
}

# HOST commands to run all CC scenarios (execute before this notebook):
print(f'CC delta source: {DELTA_SOURCE}')
print('To run all CC scenarios from the host terminal:')
for sc_name in CC_SCENARIOS:
    sc_dir = SFINCS_DIR / f'run_{sc_name}'
    print(f'  docker run --rm --platform=linux/amd64 -v {sc_dir}:/data deltares/sfincs-cpu:latest')
print()

cc_results = {}

for sc_name, delta in CC_SCENARIOS.items():
    sc_dir   = str(SFINCS_DIR / f'run_{sc_name}')
    Q_cc     = (Q_ref * delta).to_frame(name=0)

    model_cc = setup_sfincs_model(
        basin_geom       = domain_box,
        dem_path         = dem_file,
        output_dir       = sc_dir,
        discharge_series = Q_cc,
        src_points       = src_gdf,
        crs              = EPSG,
        resolution       = DX,
        manning          = 0.035,
        tref             = TREF,
        tstart           = TSTART,
        tstop            = TSTOP,
        dt_max           = 30.0,
        z_min            = -5.0,
    )
    model_cc.write()
    # WL boundary: Manning rating curve at downstream edge
    write_manning_wl_boundary(
        model_dir        = sc_dir,
        discharge_series = Q_cc,
        ch_w             = 2 * ch_half_m,
        ch_zb            = -2.5,
        mann_n           = 0.035,
        slope            = 2e-4,
        tref             = TREF,
    )

    # ── Try reading SFINCS results; fall back to Manning ─────────────────────
    nc_cc = SFINCS_DIR / f'run_{sc_name}' / 'sfincs_map.nc'
    if nc_cc.exists():
        ds_cc  = xr.open_dataset(str(nc_cc))
        hmax_cc = np.nanmax(ds_cc['hmax'].values, axis=0)
        dep_cc  = np.where(msk_v > 0, np.nan_to_num(hmax_cc), 0.0)
        area    = int((dep_cc > 0.3).sum()) * (DX / 1000) ** 2
        maxd    = float(np.nanmax(dep_cc))
        method  = 'SFINCS dynamic result'
    else:
        # Manning fallback if Docker/SFINCS output is not available; reported as approximate.
        yn_cc   = manning_yn(Q_PEAK * delta)
        wsel_cc = CH_ZB + yn_cc
        dep_cc  = np.maximum(0.0, wsel_cc - zb_dt)
        dep_cc  = np.where(msk_v > 0, dep_cc, 0.0)
        area    = int((dep_cc > 0.3).sum()) * (DX / 1000) ** 2
        maxd    = float(dep_cc.max())
        method  = 'Manning fallback (approximate)'

    cc_results[sc_name] = {'delta': delta, 'area_km2': area, 'max_depth_m': maxd, 'method': method}
    print(f'  {sc_name:<35s}  δ={delta:.2f}  area={area:.2f} km²  max={maxd:.2f} m  [{method}]')

print(f'\n{len(cc_results)} scenarios completed.')

df_cc     = pd.DataFrame(cc_results).T.sort_values('delta')
ssp_color = ['#0288D1' if 'ssp245' in n else '#C62828' for n in df_cc.index]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(df_cc.index, df_cc['area_km2'].astype(float), color=ssp_color, alpha=0.8)
axes[0].axvline(a_sf, color='black', ls='--', lw=1.5, label=f'Reference T100 ({a_sf:.1f} km²)')
axes[0].set_xlabel('Flooded area (km²)')
method_label = 'SFINCS 2D results' if any(v['method'].startswith('SFINCS') for v in cc_results.values()) else 'Manning fallback (approximate)'
axes[0].set_title(f'Illustrative CC ensemble - flood extent ({method_label})')
axes[0].legend(fontsize=9)

axes[1].scatter(df_cc['delta'].astype(float), df_cc['area_km2'].astype(float),
                c=['#0288D1' if 'ssp245' in n else '#C62828' for n in df_cc.index],
                s=80, zorder=3)
for sc, row in df_cc.iterrows():
    axes[1].annotate(sc.split('_')[0], (float(row['delta']), float(row['area_km2'])),
                     fontsize=7, ha='left', va='bottom')
axes[1].axhline(a_sf, color='black', ls='--', alpha=0.5)
axes[1].set_xlabel('Peak discharge scaling factor (δ)')
axes[1].set_ylabel('Flooded area (km²)')
axes[1].set_title('Flood area vs. discharge delta (blue=SSP2-4.5, red=SSP5-8.5)')
plt.tight_layout()
